In [1]:
INSTRUMENT_PROFILES = {
    "flute": {
        "min_pitch": 60,
        "max_pitch": 96,
        "preferred_low": 67,
        "preferred_high": 88,
        "max_polyphony": 1
    }
}


In [ ]:
# Constraint Functions
# Each note is represented as (pitch, start_time, duration)

def out_of_range(notes, profile):
    for pitch, _, _ in notes:
        if pitch < profile["min_pitch"] or pitch > profile["max_pitch"]:
            return True
    return False


def max_simultaneous_notes(notes):
    events = []
    for _, start, dur in notes:
        events.append((start, +1))
        events.append((start + dur, -1))

    events.sort()
    current = 0
    max_poly = 0

    for _, delta in events:
        current += delta
        max_poly = max(max_poly, current)

    return max_poly


def tessitura_violations(notes, profile):
    return sum(
        pitch < profile["preferred_low"] or pitch > profile["preferred_high"]
        for pitch, _, _ in notes
    )


In [ ]:
# Correction Logic: Global Octave Shifting

def shift_notes(notes, semitones):
    return [(pitch + semitones, start, dur) for pitch, start, dur in notes]


def find_feasible_transposition(notes, profile):
    for octave in range(-3, 4):
        shifted = shift_notes(notes, octave * 12)

        if not out_of_range(shifted, profile):
            if max_simultaneous_notes(shifted) <= profile["max_polyphony"]:
                return shifted, octave * 12

    return None, None


In [ ]:
# Constraint Classifier

def classify_input(notes, profile):

    polyphony = max_simultaneous_notes(notes)
    if polyphony > profile["max_polyphony"]:
        return {
            "status": "REJECTED",
            "reason": f"Polyphony {polyphony} exceeds limit {profile['max_polyphony']}",
            "notes": None
        }

    if out_of_range(notes, profile):
        corrected, shift = find_feasible_transposition(notes, profile)
        if corrected is None:
            return {
                "status": "REJECTED",
                "reason": "No feasible global transposition found",
                "notes": None
            }
        else:
            return {
                "status": "CORRECTED",
                "reason": f"Global pitch shift of {shift} semitones applied",
                "notes": corrected
            }

    return {
        "status": "ACCEPTED",
        "reason": "Input already satisfies instrument constraints",
        "notes": notes
    }


In [ ]:
# Out ofRange Flute Melody
FLUTE = {
    "min_pitch": 60,
    "max_pitch": 96,
    "preferred_low": 67,
    "preferred_high": 88,
    "max_polyphony": 1
}

melody_out_of_range = [
    (48, 0.0, 0.5),
    (50, 0.5, 0.5),
    (52, 1.0, 0.5)
]

result_1 = classify_input(melody_out_of_range, FLUTE)
result_1


{'status': 'CORRECTED',
 'reason': 'Global pitch shift of 12 semitones applied',
 'notes': [(60, 0.0, 0.5), (62, 0.5, 0.5), (64, 1.0, 0.5)]}

In [ ]:
#Chord Flute  
chord = [
    (72, 0.0, 1.0),
    (76, 0.0, 1.0),
    (79, 0.0, 1.0)
]

result_2 = classify_input(chord, FLUTE)
result_2


{'status': 'REJECTED', 'reason': 'Polyphony 3 exceeds limit 1', 'notes': None}

In [ ]:
#Valid Melody Flute 
valid_melody = [
    (72, 0.0, 0.5),
    (74, 0.5, 0.5),
    (76, 1.0, 0.5)
]

result_3 = classify_input(valid_melody, FLUTE)
result_3


{'status': 'ACCEPTED',
 'reason': 'Input already satisfies instrument constraints',
 'notes': [(72, 0.0, 0.5), (74, 0.5, 0.5), (76, 1.0, 0.5)]}